# Cookie Cats A/B Test

## Gate Placement and Player Retention

**Notebook:** 01_data_loading.ipynb  
**Phase:** 1 — Data Loading, Cleaning & Exploration  
**Source:** Kaggle "Mobile Games: A/B Testing" (`cookie_cats.csv`, 90,189 data rows), originally from DataCamp project 184  
**Author:** Taofeek Salami  
**Date:** 2026-10-16  

## Load Raw Data into BigQuery

The CSV in `data/raw/` is loaded into BigQuery unchanged as `cookie_cats.raw_cookie_cats`, using a load job (the sandbox does not allow streaming inserts). The file's SHA-256 checksum is checked first, so the load stops if the file is not the one recorded at download. Re-running the notebook replaces the table rather than appending to it, so the raw table can be rebuilt at any time.

**Column mapping notes:**

- All five column names are kept exactly as they appear in the CSV (`userid`, `version`, `sum_gamerounds`, `retention_1`, `retention_7`). They are already valid BigQuery column names, so no renaming is needed.
- All columns are loaded as `STRING` so the raw table holds the values exactly as they appear in the file. Types are checked and applied in SQL during profiling and cleaning (`sql/01_data_preparation.sql`), and values that would fail to convert are counted before any conversion.

In [1]:
import hashlib                       # computes the file's SHA-256 checksum
import os                            # reads settings from environment variables
from pathlib import Path             # builds file paths from the project root

from dotenv import load_dotenv       # loads the .env file into environment variables
from google.cloud import bigquery    # Google's Python client for BigQuery

## Configuration

Settings come from `.env`, so no project ID is written into the notebook. The expected checksum and row count are the values recorded when the file was downloaded and checked.

In [2]:
# The notebook runs from notebooks/, so the project root is one folder up.
PROJECT_ROOT = Path.cwd().resolve().parent
load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET = os.environ["BQ_DATASET"]
LOCATION = os.environ["BQ_LOCATION"]

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "cookie_cats.csv"
EXPECTED_SHA256 = "9f53027065840672e77303281289988371d4a6b67c7dcd3bd4e6306a2a263dc8"
EXPECTED_ROWS = 90_189  # header + 90,189 rows; confirmed with awk and BigQuery (wc -l shows one fewer because the file has no final newline)

# BigQuery names tables as project.dataset.table.
TABLE_ID = f"{PROJECT_ID}.{DATASET}.raw_cookie_cats"

# The client is the connection to BigQuery. It logs in with the Application
# Default Credentials created by `gcloud auth application-default login`.
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
print(TABLE_ID)

cookie-cats-ab-taofeek.cookie_cats.raw_cookie_cats


## Check the file before loading

The load stops if the file is not the one recorded at download.

In [3]:
# Hash the file and compare it with the value recorded at download.
# Any change to the file, even one character, produces a different hash.
sha256 = hashlib.sha256(RAW_FILE.read_bytes()).hexdigest()
assert sha256 == EXPECTED_SHA256, f"Checksum mismatch: {sha256}"
print("Checksum matches:", sha256)

Checksum matches: 9f53027065840672e77303281289988371d4a6b67c7dcd3bd4e6306a2a263dc8


## Load

Every column is loaded as text, with no type detection. The load still stops if the file has a structural problem, such as a row with the wrong number of columns.

In [4]:
# Every column is declared as text, so BigQuery stores the values exactly as
# they appear in the file. Types are applied later in SQL.
schema = [
    bigquery.SchemaField("userid", "STRING"),
    bigquery.SchemaField("version", "STRING"),
    bigquery.SchemaField("sum_gamerounds", "STRING"),
    bigquery.SchemaField("retention_1", "STRING"),
    bigquery.SchemaField("retention_7", "STRING"),
]

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,          # the first line is the header, not data
    schema=schema,
    autodetect=False,             # use the schema above, never guess types
    max_bad_records=0,            # fail the whole load if any row is unreadable
    # Replace the table on every run, so re-running never duplicates rows.
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# A load job uploads the whole file in one operation. BigQuery runs the job on
# its side; result() waits until it finishes and raises an error if it failed.
with RAW_FILE.open("rb") as f:     # "rb": read the file as raw bytes
    job = client.load_table_from_file(f, TABLE_ID, job_config=job_config)

job.result()
print("Load job finished:", job.job_id)   # each job has an ID you can look up in the console

Load job finished: 37ae2e17-547d-4823-98ec-735168027fdf


## Validation

The loaded table must have 90,189 rows and five `STRING` columns, and the header line must not have been loaded as a data row.

In [5]:
# get_table() fetches the table's metadata (row count, schema) without
# reading the data itself, so it costs nothing.
table = client.get_table(TABLE_ID)
print("Rows loaded:", table.num_rows)
for field in table.schema:
    print(f"  {field.name}: {field.field_type}")

assert table.num_rows == EXPECTED_ROWS, "Row count does not match the file"

# With every column loaded as text, a header line loaded by mistake would not
# cause an error, so check for it directly.
# client.query() runs the SQL in BigQuery; to_dataframe() returns the result.
header_check = client.query(f"""
    SELECT COUNTIF(userid = 'userid') AS header_rows_loaded
    FROM `{TABLE_ID}`
""").to_dataframe()

assert header_check["header_rows_loaded"][0] == 0, "Header row was loaded as data"
print("Header row not loaded as data: confirmed")

Rows loaded: 90189
  userid: STRING
  version: STRING
  sum_gamerounds: STRING
  retention_1: STRING
  retention_7: STRING
Header row not loaded as data: confirmed
